# Using `NoiseDownloader` to download noise

This example demonstrates how to download noise seismograms using the class contained in the DAE directory.

A class to handle downloading noise seismograms is available as `DAE.NoiseDownloader`, which uses ObsPy FDSN client to download traces for stations as defined. Downloaded traces undergo some basic quality control to attempt to ensure that they are noise. Documentation for this class be found in `DAE/noise.py`, but some key considerations and use cases are covered here.

Further information about ObsPy FDSN can be found here: https://docs.obspy.org/packages/obspy.clients.fdsn.html.

In [ ]:
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt

# The DAE directory can be imported as a package.
import DAE

Various choices about the dataset are made when intialising `NoiseDownloader`. Examples are given below, and the size of the files and parameters for the time axis of the traces should be consistent with any accompanying synthetic datasets downloaded with `SyntheticsDownloader`.

In [ ]:
client = 'IRIS'
traces_per_file = 6
length_of_trace_in_samples = 6001
sampling_rate_in_Hz = 20
components = 'horizontal'    # Either horizontal or vertical
STALTA = True                # Should be true to check the STA/LTA for the trace to ensure the trace is noise
output_directory = './Data/Example_Noise/'

All of these options are passed to the downloader during initialisation. If the client is not set then it defaults to 'IRIS'.

In [ ]:
nd = DAE.NoiseDownloader(client=client,
                         traces_per_file=traces_per_file,
                         length_of_trace_in_samples=length_of_trace_in_samples,
                         sampling_rate_in_Hz=sampling_rate_in_Hz,
                         components=components,
                         STALTA=STALTA,
                         output_directory=output_directory)

The `get_stations` method is essentially a wrapper for `obspy.clients.fdsn.client.Client.get_stations` and the arguments passed are essentially the same as for that function. See: https://docs.obspy.org/packages/autogen/obspy.clients.fdsn.client.Client.get_stations.html.

For example we can get the stations of the Global Seismograph Network (network codes CU, IC, II, IU), without any geographic or temporal constraints.

In [ ]:
nd.get_stations(network='CU,IC,II,IU')
print(nd.inventory)

The inbuilt methods for `obspy.core.inventory` should work. For example we could plot the stations on a map.

In [ ]:
nd.inventory.plot()

The `build_dataset` method runs the download and a progressbar keeps track. It is at this point that we say how many traces we want to download.

In [ ]:
nd.build_dataset(number_of_traces=18)

Files in .npz format containing the downloaded traces should now be in the `output_directory`.

In [ ]:
file_list = glob.glob(f'{output_directory}/*.npz')
print(file_list)

We can easily plot the contents of the files and see these traces.

In [ ]:
def plot_figure(file_list):
    # Set up the figure. Lets put each file into a different subplot
    fig, axes = plt.subplots(nrows=1, ncols=len(file_list))
    for ax, fname in zip(axes, file_list):

        # Read in the .npz file
        data = np.load(fname)['data']

        # Plot each trace
        for i, trace in enumerate(data):
            ax.plot(i + 0.5 * trace / np.max(np.abs(trace)), 'k', linewidth=0.75)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlim([0, data.shape[1]])
        ax.set_title(fname.split('/')[-1])
    plt.show()

# Call the function
plot_figure(file_list)

If we want do not want our file numbering to start from zero then we can pass `startcount` to `build_dataset`, which is the number of the file that we want to start from. This allows parallel downloads to be run without overwriting, or for existing datasets to be expanded.

In this example we will download one more file.

In [ ]:
number_of_downloaded_files = len(file_list)
nd.build_dataset(number_of_traces=6, startcount=number_of_downloaded_files)

In [ ]:
new_file_list = glob.glob(f'{output_directory}/*.npz')
plot_figure(new_file_list)